***
# OAM SEMSANS Experiment:
***

This notebook contain some theorectical calculations, McStas simulations, and the principle data analysis for the "Demonstration of orbital angular momentum in neutron beams" experiment at the HFIR at ORNL. This experiment began on August 3rd, 2023 at 3:30 pm at the HB-2D polarized development beamline during fuel cycle 502.

The experiment continued to the first part of the next cycle, cycle 503, which began on Tuesday, August 29th. The experiment ended Wednesday September 6th.


To do list:
1. [x] Check beam polarization with McStas
2. [ ] Analyze the generated OAM
3. [ ] Calculate the skyrmion number of the beam
4. [ ] Fit the pac scans to create gifs
5. [ ] Explain the beam polarization texture


## Analysis Setup
***

#### Settings, constants, and functions

In [1]:
import numpy as np

import sympy as sp

import scipy as sc
from scipy.optimize import curve_fit,minimize
from scipy.signal import hilbert,convolve,find_peaks,savgol_filter
from scipy.fft import *

import csv,warnings
from os.path import isdir,isfile
from os import mkdir,chdir
import glob

import mcstasscript as ms

import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
from matplotlib.ticker import LogLocator,LogFormatterMathtext

from mpl_toolkits.mplot3d import axes3d
from matplotlib import cm
from matplotlib.colors import LogNorm
import matplotlib.ticker as mticker
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
from copy import copy

import matplotlib.dates as mdates
from datetime import datetime,date, timedelta

from cycler import cycler
viridis = plt.colormaps['viridis']
magma = plt.colormaps['magma']
cividis = plt.colormaps['cividis']

try:
    import scienceplots
    plt.style.use(['science','notebook','grid','default','bright'])
except ImportError: 
    print('SciencePlots package not found!')

plt.rcParams.update({'font.family': 'STIXGeneral','mathtext.fontset': 'stix'})

SMALL_SIZE,MEDIUM_SIZE,BIGGER_SIZE = 22,25,28
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title
plt.rc('axes', titlesize=BIGGER_SIZE)    # actually gets title font size

def prepare_plot(plot_parms=None):
    '''Helper function that returns basic figure object.'''
    DEFAULT_PLOT_PARMS = {'size':(10,6),'title':[None],'xlabel':[None],'ylabel':[None],\
                          'grid':False,'figs':(1,1),'suptitle':None,'repeat':True}
    if plot_parms is None: plot_parms = {}
    parms = DEFAULT_PLOT_PARMS|plot_parms
    
    fig, axs = plt.subplots(*parms['figs'], figsize=parms['size'])
    axs = np.asarray(axs,dtype=object).reshape(-1)

    fig_num = int(parms['figs'][0]*parms['figs'][1])
    plt.suptitle(parms['suptitle'])

    if parms['repeat']:
        parms['title'] = parms['title']*fig_num
        parms['xlabel'] = parms['xlabel']*fig_num
        parms['ylabel'] = parms['ylabel']*fig_num

    for indx,ax in enumerate(axs):
        ax.set(title=parms['title'][indx],
               xlabel=parms['xlabel'][indx],
               ylabel=parms['ylabel'][indx])
        ax.grid(parms['grid'])

    return fig,axs


In [2]:
LAM = 4.25  #HB-2D wavelength
DLAM = LAM*1/100  #HB-2D wavelength spread
MWP_GPA = 35.  #MWP gauss per amp
NU_GPA = 5.  #Nutator gauss per amp
gn = -1.832471e8  #gyromagnetic ratio of neutron (rad/T/m^2)
mn = 1.674927e-27  #mass of neutron (kg)
hbar = 1.054571e-34  #reduced plancks constant (kg m^2/s)
cl_si = gn/(hbar*2*np.pi)*mn  #Larmor constant (SI units)
cl = cl_si*1e-14
print(f'Larmor constant = {round(cl,3)} (G^-1 m^-1 A^-1)')

pix_to_mm = .08287
print(f'Timepix3 pixel size = {round(pix_to_mm*1e3)} (um)')

ac_2_period = 4.4788  #period from fit at 4.25 A
AC_GPA = abs(2*np.pi/(ac_2_period*cl*LAM*.015))
print(f'ac_2 pi/2 condition: {round(ac_2_period/4,3)} amps')
print(f'AC gpa = {round(AC_GPA,3)} (G/Amp)')


Larmor constant = -4.632 (G^-1 m^-1 A^-1)
Timepix3 pixel size = 83 (um)
ac_2 pi/2 condition: 1.12 amps
AC gpa = 4.751 (G/Amp)


### Calculating the SEMSANS conditions

In [ ]:
def get_period(I1,I2,gpa=MWP_GPA,lam=LAM,show=False):
    """Returns SEMSANS period for the MWPs."""
    assert I2 > I1, 'Second MWP current must be larger than the first!'
    result = np.pi/(cl*gpa*lam*(I1-I2))
    if show: print(f'Period: {np.round(result*1e3,2)} mm')
    return result

def get_sem(B1,L1,L2,show=False):
    """Returns focusing condition for prism in regular SEMSANS."""
    assert L1 >= L2, 'L1 > L2!'
    result = B1*L1/L2
    if show: print(f'SEMSANS focused current: {round(result,2)} amps')
    return result

def get_ch(B1,L1,L2,L3,L4,show=False):
    """Returns focusing conditions for 4 MWPs in checkerboard SEMSANS."""
    assert L1 >= L2 or L2 >= L3 or L3 >= L4, 'Distances must be in strict descending order!'
    result = np.array([-B1*L1/L2, B1*(L1 - L2)/(L3 - L4)*L4/L2, -B1*(L1 - L2)/(L3 - L4)*L3/L2])
    if show: print(f'2D SEMSANS focused current: {np.round(result,2)} amps')
    return result

def cos(x,amp,freq,phase,shim,offset):
    """Returns a cosine with ampitude, frequency, phase, shim, and offset."""
    return amp*np.cos(freq*(x - offset) + phase) + shim

def get_flip(dist,gpa=AC_GPA,lam=LAM,flip=np.pi,show=True):
    """Returns the current required for pi or pi/2 flip."""
    result = round(flip/(cl*gpa*lam*dist*1e-2),2)
    if show: print(f'Current required for a {round(flip,2)} flip: {result} amps')
    return result

def frp_convert(value,show=False):
    """Convert polarization to flipping ratio and vice-versa."""
    assert value != 1, 'FR is infinite when polarization is unity!'
    if value > 1:
        result = (value - 1)/(value + 1)
        something = f'FR: {round(value,3)} \n Pol: {round(result*100,1)}%'
    else:
        result = (1 + value)/(1 - value)
        something = f'FR: {round(result,3)} \n Pol: {round(value*100,1)}%'
    if show: print(something)
    return result

In [ ]:
tin,pin,kx,ky,x,y = sym.symbols(r'\theta_{in},\phi_{in},k_x,k_y,x,y',real=True)

Polx = sym.sin(tin)*sym.cos(pin)*sym.cos(ky*y) + sym.sin(kx*x)*(sym.cos(tin)*sym.cos(ky*y) - sym.sin(tin)*sym.sin(pin)*sym.sin(ky*y))
display(Polx)
Poly = sym.sin(tin)*sym.sin(pin)*sym.cos(ky*y) + sym.cos(tin)*sym.sin(ky*y)
display(Poly)
Polz = -sym.sin(tin)*sym.cos(pin)*sym.sin(kx*x) + sym.cos(kx*x)*(sym.cos(tin)*sym.cos(ky*y) - sym.sin(tin)*sym.sin(pin)*sym.sin(ky*y))
display(Polz)

print('\n Pol_in = (0,1,0):')
Polx = Polx.subs({tin:sym.pi/2,pin:sym.pi/2})
display(Polx.expand().simplify())
Poly = Poly.subs({tin:sym.pi/2,pin:sym.pi/2})
display(Poly.expand().simplify())
Polz = Polz.subs({tin:sym.pi/2,pin:sym.pi/2})
display(Polz.expand().simplify())

print('\n Beam pol:')
PB = sym.sqrt(Polx**2 + Poly**2 + Polz**2)
PB = PB.subs({tin:sym.pi/2,pin:-sym.pi/2})
display(PB.expand().simplify())

Polx = sym.sin(tin)*sym.cos(pin)*sym.cos(ky*y) + sym.sin(kx*x)*(sym.cos(tin)*sym.cos(ky*y) - sym.sin(tin)*sym.sin(pin)*sym.sin(ky*y))
Poly = sym.sin(tin)*sym.sin(pin)*sym.cos(ky*y) + sym.cos(tin)*sym.sin(ky*y)
Polz = -sym.sin(tin)*sym.cos(pin)*sym.sin(kx*x) + sym.cos(kx*x)*(sym.cos(tin)*sym.cos(ky*y) - sym.sin(tin)*sym.sin(pin)*sym.sin(ky*y))

print('\n Pol_in = (0,1/sqrt(2),1/sqrt(2)):')
Polx = Polx.subs({tin:sym.pi/4,pin:sym.pi/2})
display(Polx.expand().simplify())
Poly = Poly.subs({tin:sym.pi/4,pin:sym.pi/2})
display(Poly.expand().simplify())
Polz = Polz.subs({tin:sym.pi/4,pin:sym.pi/2})
display(Polz.expand().simplify())

print('\n Beam pol:')
PB = sym.sqrt(Polx**2 + Poly**2 + Polz**2)
PB = PB.subs({tin:sym.pi/4,pin:sym.pi/2})
display(PB.expand().simplify())


#### Cycle 502 measurements

In [ ]:
LM1 = 1.5
LM2 = 1.09

L1 = LM1 + .047
L2 = LM1 - .063

L3 = LM2 + .067
L4 = LM2 - .045

In [ ]:
B1 = 9.29
_ = get_period(B1,get_sem(B1,L1,L2,show=True),show=True)
#_ = get_ch(B1,1.5,1.5-.13,1.09+.07,1.09-.07,show=True)  #used this wrong SEMSANS condition for the 10 amp runs
_ = get_ch(B1,L1,L2,L3,L4,show=True)

In [ ]:
B1 = 15
_ = get_period(B1,get_sem(B1,L1,L2,show=True),show=True)
_ = get_ch(B1,L1,L2,L3,L4,show=True)

In [ ]:
B1 = 20
_ = get_period(B1,get_sem(B1,L1,L2,show=True),show=True)
_ = get_ch(B1,L1,L2,L3,L4,show=True)

In [ ]:
#Fumi moved detector
L1 = LM1 + .047 + .43
L2 = LM1 - .063 + .43

L3 = LM2 + .067 + .43
L4 = LM2 - .045 + .43

In [ ]:
B1 = 2
_ = get_period(B1,get_sem(B1,L1,L2,show=True),show=True)
_ = get_ch(B1,L1,L2,L3,L4,show=True)

#### Cycle 503 measurements

In [ ]:
LM1 = 1.5  #check these
LM2 = 1.08

L1 = LM1 + .047
L2 = LM1 - .06375

L3 = LM2 + .0665
L4 = LM2 - .0445

In [ ]:
B1 = 20.
_ = get_period(B1,get_sem(B1,L1,L2,show=True),show=True)
_ = get_ch(B1,L1,L2,L3,L4,show=True)

In [ ]:
B1 = 20.
_ = get_period(B1,get_sem(B1,L3,L4,show=True),show=True)

In [ ]:
B1 = 20.
_ = get_period(B1,get_sem(B1,L1,L2,show=True),show=True)

### Spin echo length correction

In [9]:
%%time

# Symbols
x0, theta, theta0, B, Bg, mu, k = sp.symbols('x0 theta theta0 B Bg mu k', real=True)
L1, L2, Ls, a, lam = sp.symbols('L1 L2 Ls a lambda', positive=True)
B1, B2 = sp.symbols('B1 B2', nonnegative=True)
c = sp.symbols('c', negative=True)
pi = sp.pi

# Helper functions
def Indx(*args):
    """Neutron refractive index for a magnetic field configuration."""
    if len(args) == 4:
        wavelength, b1, b2, mu = args
        return 1 - mu*wavelength**2/(4*pi)*c*(b1-b2)
    if len(args) == 3:
        b1, b2, mu = args
        return 1 + mu*k*(b1-b2)
    raise ValueError("Indx expects either 3 or 4 arguments")

def p(p0, n, l0, l):
    """Intersection of a ray with a line."""
    p0, n, l0, l = map(sp.Matrix, (p0, n, l0, l))
    return l0 + l*(p0-l0).dot(n)/l.dot(n)

def series(expr, var, order):
    """Taylor expansion about var = 0."""
    expand = lambda x: sp.series(x, var, 0, order+1).removeO()
    return expr.applyfunc(expand) if isinstance(expr, sp.MatrixBase) else expand(expr)

def series_2d(expr, angle=theta0, angle_order=1, wavelength_order=2):
    """Sequential Taylor expansion in angle and wavelength."""
    def expand(x):
        x = sp.series(x, angle, 0, angle_order+1).removeO()
        return sp.series(x, lam, 0, wavelength_order+1).removeO()
    return expr.applyfunc(expand) if isinstance(expr, sp.MatrixBase) else expand(expr)

def print_result(label, expr):
    """Print a labeled symbolic expression."""
    print(f"\n{'='*70}")
    print(label)
    print('='*70)
    print(sp.sstr(expr))

# Initial angular expansion
initial = pi/4 - sp.asin(sp.sin(pi/4-theta)*Indx(lam, B, -B, mu))
initial_expansion = sp.simplify(series_2d(initial, theta, 1, 2))

# Refractive indices
Indx01 = Indx(lam, 0, B1, mu)
Indx12 = Indx(lam, B1, -B1, mu)
Indx23 = Indx(lam, -B1, 0, mu)
Indx34 = Indx(lam, 0, -B2, mu)
Indx45 = Indx(lam, -B2, B2, mu)
Indx56 = Indx(lam, B2, 0, mu)

# Geometry
edge = [0, 1]
hyp = [-1, 1]
det = [0, 1]

# First prism
p01 = p([0, -a/2], edge, [x0, -a/2], [sp.sin(theta0), sp.cos(theta0)])
theta_1 = series(theta0*Indx01, lam, 2)

p12 = series_2d(p([0, 0], hyp, p01, [sp.sin(theta_1), sp.cos(theta_1)]))
theta_2 = series_2d(sp.asin(sp.sin(theta_1+pi/4)*Indx12)-pi/4)

p23 = series_2d(p([0, a/2], edge, p12, [sp.sin(theta_2), sp.cos(theta_2)]))
theta_3 = series(theta_2*Indx23, lam, 2)

# Second prism
p34 = series_2d(p([0, L1-L2-a/2], edge, p23, [sp.sin(theta_3), sp.cos(theta_3)]))
theta_4 = series(theta_3*Indx34, lam, 2)

p45 = series_2d(p([0, L1-L2], hyp, p34, [sp.sin(theta_4), sp.cos(theta_4)]))
theta_5 = series_2d(sp.asin(sp.sin(theta_4+pi/4)*Indx45)-pi/4)

p56 = series_2d(p([0, L1-L2+a/2], edge, p45, [sp.sin(theta_5), sp.cos(theta_5)]))
theta_6 = series(theta_5*Indx56, lam, 2)

# Detector
pdet = series_2d(p([0, L1-Ls], det, p56, [sp.sin(theta_6), sp.cos(theta_6)]))


CPU times: total: 1min 2s
Wall time: 1min 4s


In [10]:
# Spin-dependent displacement
PlusState = pdet.subs(mu, 1)
MinusState = pdet.subs(mu, -1)
delta = PlusState - MinusState

# Signed horizontal separation
DeltaX = sp.factor(delta[0])
DeltaY = sp.factor(delta[1])

# Collect angular contributions
DeltaX = sp.collect(sp.expand(DeltaX), theta0)
DeltaY = sp.collect(sp.expand(DeltaY), theta0)

# Separation magnitude
Xi = sp.sqrt(DeltaX**2 + DeltaY**2)

# SESANS and SEMSANS
Xi_SES = sp.simplify(Xi.subs(B1, B2))
Xi_SEM = sp.simplify(Xi.subs(B1, B2*L2/L1))

# Normal incidence
Xi_SES_normal = sp.simplify(Xi_SES.subs(theta0, 0))
Xi_SEM_normal = sp.simplify(Xi_SEM.subs(theta0, 0))

print_result("SESANS: General incidence", Xi_SES)
print_result("SESANS: Normal incidence", Xi_SES_normal)

print_result("SEMSANS: General incidence", Xi_SEM)
print_result("SEMSANS: Normal incidence", Xi_SEM_normal)



SESANS: General incidence
-B2*c*lambda**2*Abs(L1 - L2 + 2*theta0*(L1 - L2))/pi

SESANS: Normal incidence
-B2*c*lambda**2*Abs(L1 - L2)/pi

SEMSANS: General incidence
-B2*c*lambda**2*Abs(2*L1*(Ls + x0) - 2*L2*(Ls + x0) + theta0*(L1*(2*L1 - 2*L2 + 2*Ls + a + 6*x0) - L2*(2*Ls + a + 6*x0)))/(2*pi*L1)

SEMSANS: Normal incidence
-B2*c*lambda**2*Abs((L1 - L2)*(Ls + x0))/(pi*L1)


***
## McStas Simulations
***

Simulated experimental configurations.

### Building the instrument file

The action of the v-cavities, nutators, and field stepper are not currently taken in account. The divergence is estimated by the most divergent neutron that could pass through both of the v-cavities. 

In [ ]:
lambda0 = LAM  #mean wavelength (Angstroms)
if True: dlambda = DLAM  #wavelength spread (Angstroms)
else: dlambda = 1e-20

ap_width,ap_height = 20.,20.  #rectangular aperture of v-cavity (mm)

MWP_str = 2*MWP_GPA*1e-4  #device field strengths (T/Amp)
#NU_str = NU_GPA*1e-4
#CRYO_str = AC_GPA*1e-4

MWP_dims = [.02,.02,.105]  #dimension in meters (x,y,z)
#NU_dims = [.03,.03,.03]
#AC_dims = [.03,.03,.014]
#FS_dims = [.05,.05,.12]

MWP_sep = 0.75e-3
#NU_sep = .05
#CYRO_sep = .005

MWP_angles = [0.,0.,-90.,-90.]

d1 = .82
d2 = d1 + .111
d3 = 1.24
d4 = d3 + .111
Ld = .78 + 2*0.3 + .12 + .92
dists = {'nu1':None,'ac1':None,'MWP1':d1,'MWP2':d2,'MWP3':d3,'MWP4':d4,'ac2':None,'nu2':None,'det':Ld}
L1,L2,L3,L4 = Ld - np.array([d1 - MWP_dims[2]/2,d2 - MWP_dims[2]/2,d3 - MWP_dims[2]/2,d4 - MWP_dims[2]/2])

c1 = 20.
#c2,c3,c4 = get_ch(c1,L1,L2,L3,L4,show=True)
c2 = -1*get_sem(c1,L1,L2,show=False)
c3 = 20.
c4 = -1*get_sem(c3,L3,L4,show=False)
print('First SEMSANS period:')
_ = get_period(c1,abs(c2),show=True)
print('Second SEMSANS period:')
_ = get_period(c3,abs(c4),show=True)
print('\n')

print(f'Wavelength: {lambda0} p/m {round(dlambda,3)}')
print(f'Aperture (mm): {ap_width,ap_height}')
print(f'Front distances from MWP to source (m): {np.round([d1,d2,d3,d4],3)}')
print(f'Center distances MWP to detector (m): {np.round([L1,L2,L3,L4],3)}')

pol_dir = [1,0,0]
pol_dir = [i/np.sqrt(sum([j**2 for j in pol_dir])) for i in pol_dir]
print(f'Polarizer axis: {[round(i,3) for i in pol_dir]}')

det_x,det_y = 20.,20.
det_bins = 256

print(f'Detector size (mm): {np.round([det_x,det_y],3)}')
print(f'[x,y] bin size (mm): {np.round([det_x/det_bins,det_y/det_bins],3)} \n')

int_det_file = "\"int_det.dat\""
pol_det_files = [f"\"pol{i}_det.dat\"" for i in ["x","y","z"]]
mon_det_files = [f"\"mon{i}_det.dat\"" for i in ["x","y","z"]]

print("-"*80)  #Start of instrument definition
OAM = ms.McStas_instr('OAM',author='Sam McKay',origin='Indiana University')
delta_lambda = OAM.add_parameter("double","delta_lambda",value=dlambda,comment="Wavelength spread [AA]")
#ac1 = OAM.add_parameter("double","ac1",value=0.,comment="Accelerator coil 1 field [Gauss]")
#ac2 = OAM.add_parameter("double","ac2",value=0.,comment="Accelerator coil 2 field [Gauss]")
cur1 = OAM.add_parameter("double","cur1",value=c1,comment="MWP1 current [Amps]")
cur2 = OAM.add_parameter("double","cur2",value=c2,comment="MWP2 current [Amps]")
cur3 = OAM.add_parameter("double","cur3",value=c3,comment="MWP3 current [Amps]")
cur4 = OAM.add_parameter("double","cur4",value=c4,comment="MWP4 current [Amps]")
cur1 = OAM.add_parameter("double","ang1",value=MWP_angles[0],comment="MWP1 angle [deg]")
cur2 = OAM.add_parameter("double","ang2",value=MWP_angles[1],comment="MWP2 angle [deg]")
cur3 = OAM.add_parameter("double","ang3",value=MWP_angles[2],comment="MWP3 angle [deg]")
cur4 = OAM.add_parameter("double","ang4",value=MWP_angles[3],comment="MWP4 angle [deg]")
#nu1 = OAM.add_parameter("double","nu1",value=0.,comment="Nutator 1 field [Gauss]")
#nu2 = OAM.add_parameter("double","nu2",value=0.,comment="Nutator 2 field [Gauss]")
#fs = OAM.add_parameter("double","fs",value=0.,comment="Field stepper field [Gauss]")
polx = OAM.add_parameter("double","polx",value=pol_dir[0],comment="x-component of polarizer [N/A]")
poly = OAM.add_parameter("double","poly",value=pol_dir[1],comment="y-component of polarizer [N/A]")
polz = OAM.add_parameter("double","polz",value=pol_dir[2],comment="z-component of polarizer [N/A]")
OAM.show_parameters()

print("-"*80)
OAM.add_component("origin","Progress_bar")
OAM.add_component("arm","Arm",AT=[0,0,0],RELATIVE="origin")

src = OAM.add_component("src","Source_gen",AT=[0,0,0],RELATIVE="origin")
if True:
    src.set_parameters(xwidth=ap_width*1e-3,yheight=ap_height*1e-3,I1=1e12,lambda0=lambda0,dlambda="delta_lambda",\
                       focus_xw=det_x*1e-3,focus_yh=det_y*1e-3,dist=Ld) 
else:
    src.set_parameters(xwidth=ap_width*1e-3,yheight=ap_height*1e-3,I1=1e12,lambda0=lambda0,dlambda="delta_lambda",\
                       focus_aw=1e-20,focus_ah=1e-20,dist=Ld)  #turn off divergence
print(src)

pol = OAM.add_component("pol","Set_pol",AT=[0,0,0],RELATIVE="origin")
pol.set_parameters(px="polx",py="poly",pz="polz")
print(pol)

if False:
    print("-"*80)
    NU1 = OAM.add_component("NU1","Pol_constBfield", AT=[0,0,dists['nu1']], RELATIVE="origin")
    NU1.set_parameters(xwidth=NU_dims[0],yheight=NU_dims[1],zdepth=NU_dims[2],B="nu1")
    print(NU1)
    AC1 = OAM.add_component("AC1","Pol_constBfield", AT=[0,0,dists['ac1']], RELATIVE="origin")
    AC1.set_parameters(xwidth=AC_dims[0],yheight=AC_dims[1],zdepth=AC_dims[2],B="ac1")
    print(AC1)

if True:
    print("-"*80)
    MWP1 = OAM.add_component("MWP1","Pol_MWP_v5", AT=[0,0,dists['MWP1']],RELATIVE="origin")
    MWP1.set_parameters(xwidth=MWP_dims[0],yheight=MWP_dims[1],zdepth=MWP_dims[2],current="cur1")
    MWP1.set_ROTATED([0,0,"ang1"],RELATIVE="ABSOLUTE")
    MWP2 = OAM.add_component("MWP2","Pol_MWP_v5", AT=[0,0,dists['MWP2']],RELATIVE="origin")
    MWP2.set_parameters(xwidth=MWP_dims[0],yheight=MWP_dims[1],zdepth=MWP_dims[2],current="cur2")
    MWP2.set_ROTATED([0,0,"ang2"],RELATIVE="ABSOLUTE")
    print(MWP1,'\n',MWP2)
    
if False:
    diff = (dists['MWP3'] - MWP_dims[2] - dists['MWP2'])/2
    mon_x = OAM.add_component("mon_x","Monitor_nD",AT=[0,0,dists['MWP2']+MWP_dims[2]+diff],RELATIVE="origin")
    mon_x.set_parameters(options=f'"signal=sx, x bins {det_bins}, y bins {det_bins}"',\
                         filename=mon_det_files[0],xwidth=det_x*1e-3,yheight=det_y*1e-3)
    mon_x.set_ROTATED([0,0,0],RELATIVE="ABSOLUTE")
    print(mon_x)
    mon_y = OAM.add_component("mon_y","Monitor_nD",AT=[0,0,dists['MWP2']+MWP_dims[2]+diff],RELATIVE="origin")
    mon_y.set_parameters(options=f'"signal=sy, x bins {det_bins}, y bins {det_bins}"',\
                         filename=mon_det_files[1],xwidth=det_x*1e-3,yheight=det_y*1e-3)
    mon_y.set_ROTATED([0,0,0],RELATIVE="ABSOLUTE")
    print(mon_y)
    mon_z = OAM.add_component("mon_z","Monitor_nD",AT=[0,0,dists['MWP2']+MWP_dims[2]+diff],RELATIVE="origin")
    mon_z.set_parameters(options=f'"signal=sz, x bins {det_bins}, y bins {det_bins}"',\
                         filename=mon_det_files[2],xwidth=det_x*1e-3,yheight=det_y*1e-3)
    mon_z.set_ROTATED([0,0,0],RELATIVE="ABSOLUTE")
    print(mon_z)
    
if True:
    MWP3 = OAM.add_component("MWP3","Pol_MWP_v5", AT=[0,0,dists['MWP3']],RELATIVE="origin")
    MWP3.set_parameters(xwidth=MWP_dims[0],yheight=MWP_dims[1],zdepth=MWP_dims[2],current="cur3")
    MWP3.set_ROTATED([0,0,"ang3"],RELATIVE="ABSOLUTE")
    MWP4 = OAM.add_component("MWP4","Pol_MWP_v5", AT=[0,0,dists['MWP4']], RELATIVE="origin")
    MWP4.set_parameters(xwidth=MWP_dims[0],yheight=MWP_dims[1],zdepth=MWP_dims[2],current="cur4")
    MWP4.set_ROTATED([0,0,"ang4"],RELATIVE="ABSOLUTE")
    print(MWP3,'\n',MWP4)

if False:
    print("-"*80)
    AC2 = OAM.add_component("AC2","Pol_constBfield", AT=[0,0,dists['ac2']], RELATIVE="origin")
    AC2.set_parameters(xwidth=AC_dims[0],yheight=AC_dims[1],zdepth=AC_dims[2],B="ac2")
    AC2.set_ROTATED([0,0,90],RELATIVE="ABSOLUTE")
    print(AC2)
    NU2 = OAM.add_component("NU2","Pol_constBfield", AT=[0,0,dists['nu2']], RELATIVE="origin")
    NU2.set_parameters(xwidth=NU_dims[0],yheight=NU_dims[1],zdepth=NU_dims[2],B="nu2")
    NU2.set_ROTATED([0,0,90],RELATIVE="ABSOLUTE")
    print(NU2)

print("-"*80)
pol_x = OAM.add_component("pol_x","Monitor_nD",AT=[0,0,dists['det']],RELATIVE="origin")
pol_x.set_parameters(options=f'"signal=sx, x bins {det_bins}, y bins {det_bins}"',\
                     filename=pol_det_files[0],xwidth=det_x*1e-3,yheight=det_y*1e-3)
pol_x.set_ROTATED([0,0,0],RELATIVE="ABSOLUTE")
print(pol_x)
pol_y = OAM.add_component("pol_y","Monitor_nD",AT=[0,0,dists['det']],RELATIVE="origin")
pol_y.set_parameters(options=f'"signal=sy, x bins {det_bins}, y bins {det_bins}"',\
                     filename=pol_det_files[1],xwidth=det_x*1e-3,yheight=det_y*1e-3)
pol_y.set_ROTATED([0,0,0],RELATIVE="ABSOLUTE")
print(pol_y)
pol_z = OAM.add_component("pol_z","Monitor_nD",AT=[0,0,dists['det']],RELATIVE="origin")
pol_z.set_parameters(options=f'"signal=sz, x bins {det_bins}, y bins {det_bins}"',\
                     filename=pol_det_files[2],xwidth=det_x*1e-3,yheight=det_y*1e-3)
pol_z.set_ROTATED([0,0,0],RELATIVE="ABSOLUTE")
print(pol_z)
int_det = OAM.add_component("int_det","PSD_monitor",AT=[0,0,dists['det']],RELATIVE="origin")
int_det.set_parameters(nx=det_bins,ny=det_bins,filename=int_det_file,xwidth=det_x*1e-3,yheight=det_y*1e-3)
int_det.set_ROTATED([0,0,0],RELATIVE="ABSOLUTE")
print(int_det)

print("-"*80)    #Final instrument summary:
OAM.show_components()

### Simulation scans

In [ ]:
sim_directories = {'work':'C://Users//xsm//Documents//GitHub//OAM-HFIR-11-22//sim_data//',\
                   'home':'C://Users\samck//Documents//GitHub//OAM-HFIR-11-22//sim_data//',\
                   'laptop':'C://Users//samck//Documents//GitHub//OAM-HFIR-11-22//sim_data//'}

DEFAULT_DIRECT = sim_directories['home']
assert isdir(DEFAULT_DIRECT), 'Default directory is not found!'
print(f'Default directory: {DEFAULT_DIRECT}')

QUICK_RUN,GOOD_RUN,GREAT_RUN = 1e5,1e6,1e7

def do_run(folder,fname,ray_num=QUICK_RUN,direct=DEFAULT_DIRECT,plot=False,plot_indx=0):
    """Performs McStas simulation. Returned data is a list of four McStas data objects.
    The last three are the polarization components."""
    if not isdir(f'{direct}//{folder}//'):
        mkdir(f'{direct}//{folder}//')  #makes directory if not found
    OAM.settings(output_path=f'{direct}//{folder}//{fname}',ncount=ray_num)
    data = OAM.backengine()
    if plot:
        try:
            plot_mcstas(data,proj=True,plot_indx=plot_indx)
        except Exception:
            print('Plot failed! Check plot index.')
    return data

def plot_mcstas(data,nu_post_angle=0.,proj=False,texture=False,\
                texture_parms={'plot_z':True,'plot_t':True,'ROI':[100,200,100,200],'sf':4,'in_mm':True},\
                detlims=None,plot_indx=0):
    """Plots result from list of McStas data objects or numpy arrays."""
    try:
        px,py,pz = [data[i].Intensity for i in range(plot_indx,plot_indx+3)]
        px,py = nu_post(px,py,nu_post_angle=nu_post_angle)
        lims = np.array(list(map(float,data[1].metadata.info['xylimits'].split())))*100.  #convert from meters to mm
    except Exception:
        px,py,pz = data  #reading from numpy array
        lims = detlims*100
        
    assert np.shape(px) == np.shape(py) == np.shape(pz), 'Polarization components must have the same shape!'
    
    if proj:
        fig,ax = plt.subplots(nrows=1,ncols=3,figsize=(18,5))
        im1 = ax[0].imshow(px,origin='lower',extent=lims)
        ax[0].set_title('Pol x')
        plt.colorbar(im1,ax=ax[0],shrink=.75)
    
        im2 = ax[1].imshow(py,origin='lower',extent=lims)
        ax[1].set_title('Pol y')
        plt.colorbar(im2,ax=ax[1],shrink=.75)
    
        im3 = ax[2].imshow(pz,origin='lower',extent=lims)
        ax[2].set_title('Pol z')
        plt.colorbar(im3,ax=ax[2],shrink=.75)
        fig.supxlabel('x [cm]')
        fig.supylabel('y [cm]')
        plt.tight_layout()
        plt.show()

    if texture:
        if texture_parms['ROI'] == None:
            start,stop = (0,0),(len(px),len(px))  #use entire detector
        else:
            sx,tx,sy,ty = texture_parms['ROI']
            start,stop = (sx,sy),(tx,ty)
        x_points = np.arange(start[0],stop[0],texture_parms['sf'])
        y_points = np.arange(start[1],stop[1],texture_parms['sf'])

        fig,ax = plt.subplots(1,1,figsize=(9,9))
        if texture_parms['in_mm']:
            x_points = (x_points - x_points[len(x_points)//2])*pix_to_mm  #global parameter
            y_points = (y_points - y_points[len(y_points)//2])*pix_to_mm
            ax.set_xlabel('x [mm]')
            ax.set_ylabel('y [mm]')
            
        if texture_parms['plot_z']:
            imz = ax.imshow(pz[start[0]:stop[0],start[1]:stop[1]],extent=[x_points[0],x_points[-1],y_points[0],y_points[-1]],\
                            cmap='GnBu',interpolation='none',origin='lower',zorder=0) #YlOrRd GnBu
            plt.colorbar(imz,ax=ax,shrink=.7)
        if texture_parms['plot_t']:
            QX,QY = np.meshgrid(x_points,y_points)
            ax.quiver(QX,QY,px[start[0]:stop[0]:texture_parms['sf'],start[1]:stop[1]:texture_parms['sf']],\
                      py[start[0]:stop[0]:texture_parms['sf'],start[1]:stop[1]:texture_parms['sf']],\
                      scale=20,scale_units='width',color='white',pivot='tail',\
                      width=3e-3,headwidth=5,ls='solid',linewidth=1.25,edgecolor='black',zorder=20)
        ax.set_title(r'Spin Texture: $\langle \sigma(x,y) \rangle$')
        plt.tight_layout()
        plt.show()

def nu_post(px,py,nu_post_angle=0.):
    """Chooses the orientation of the final nutator."""
    px_new = px*np.cos(nu_post_angle) - py*np.sin(nu_post_angle)
    py_new = px*np.sin(nu_post_angle) + py*np.cos(nu_post_angle)
    return px_new,py_new
    
def read_run(path,file,data_rng=(41,297),err_rng=(298,554),lim_num=38):
    """Reads run into from McStas data file."""
    filename = path + "\\" + file
    assert isdir(path), f'Data directory {path} is not found!'
    assert isfile(filename), f'Datafile {filename} is not found!'
    with open(filename, 'r') as file:
        lines = file.readlines()
        data = np.array([(l.strip()).split(' ') for l in lines[slice(*data_rng)]],dtype=float)
        errors = np.array([(l.strip()).split(' ') for l in lines[slice(*err_rng)]],dtype=float)
        lims = np.array(((lines[lim_num].strip()).split(':')[1]).split(' ')[1:],dtype=float)
    return data,errors,lims

def get_beam_pol(mcstas_data):
    '''Returns beam polarization and detector extent from a McStas data object or numpy array.'''
    try:
        beam_pol = np.sqrt(mcstas_data[0].Intensity**2 + mcstas_data[1].Intensity**2 + mcstas_data[2].Intensity**2)
        limits = square[1].metadata.limits
    except Exception:
        beam_pol = np.sqrt(mcstas_data[0]**2 + mcstas_data[1]**2 + mcstas_data[2]**2)
        limits = None
    return beam_pol,limits

def ac1_ac2_scan(folder,Bac1,Bac2,ray_num=QUICK_RUN,direct=DEFAULT_DIRECT):
    """Performs a complete ac_1 ac_2 scan. Returns a list of lists of McStas data objects."""
    if isinstance(Bac1,float):
        Bac1 = [Bac1]  #allows for single-number input
    if isinstance(Bac2,float):
        Bac1 = [Bac2]
    result = []
    for c1 in Bac1:
        for c2 in Bac2:
            ac1.value = c1
            ac2.value = c2
            result.append(do_run(folder,f'ac1_{round(c1,1)}_ac2_{round(c2,1)}',ray_num=ray_num,direct=direct,plot=False))
    ac1.value = 0.  #return to zero
    ac2.value = 0.
    return result

def plot_st(polx,poly,polz,plot_z=True,plot_t=True,start=(0,0),stop=(249,249),sf=4,vals=(-1,1),in_mm=False,\
            plot_err=True,errx=None,erry=None,errz=None,\
            plot_ana=False,phi=0,theta=np.pi/2):
    """Plots the spin texture from the polarization data. Scale factor (sf) must be an integer."""
    assert (start[0] < stop[0]) and (start[1] < stop[1]), f'The start pixel must be before the stop pixel!'
    assert (stop[0] < len(polx)) and (stop[0] < len(polx)), f'X stop must be less than the length of the data! ({len(polx)})'
    assert (stop[1] < len(poly)) and (stop[1] < len(polx)), f'Y stop must be less than the length of the data! ({len(poly)})'
    
    x_points = np.arange(start[0],stop[0],sf)
    y_points = np.arange(start[1],stop[1],sf)
    if in_mm:
        x_points = (x_points - x_points[len(x_points)//2])*pix_to_mm
        y_points = (y_points - y_points[len(y_points)//2])*pix_to_mm
    
    fig,ax = plt.subplots(1,1,figsize=(10,8))
    if plot_z:
        imz = ax.imshow(polz[start[0]:stop[0],start[1]:stop[1]],extent=[x_points[0],x_points[-1],y_points[0],y_points[-1]],\
                        cmap='GnBu',interpolation='none',origin='lower',zorder=0,vmin=vals[0],vmax=vals[1]) #YlOrRd GnBu
        plt.colorbar(imz,ax=ax,shrink=.5)
    if plot_t:
        QX,QY = np.meshgrid(x_points,y_points)
        ax.quiver(QX,QY,polx[start[0]:stop[0]:sf,start[1]:stop[1]:sf],poly[start[0]:stop[0]:sf,start[1]:stop[1]:sf],\
                  scale=20,scale_units='width',color='white',pivot='tail',\
                  width=3e-3,headwidth=5,ls='solid',linewidth=1.25,edgecolor='black',zorder=20)
    ax.set_title(r'Spin Texture: $\langle \sigma(x,y) \rangle$')
    plt.tight_layout()
    returned = [fig,ax]

    if plot_err:
        fig,ax = plt.subplots(1,1,figsize=(10,8))
        if plot_z:
            imz = ax.imshow(errz[start[0]:stop[0],start[1]:stop[1]],extent=[x_points[0],x_points[-1],y_points[0],y_points[-1]],\
                            cmap='GnBu',interpolation='none',origin='lower',zorder=0,vmin=0,vmax=np.max(errz[start[0]:stop[0],start[1]:stop[1]]))
            plt.colorbar(imz,ax=ax,shrink=.5)
        if plot_t:
            QX,QY = np.meshgrid(x_points,y_points)
            ax.quiver(QX,QY,errx[start[0]:stop[0]:sf,start[1]:stop[1]:sf],erry[start[0]:stop[0]:sf,start[1]:stop[1]:sf],\
                      scale=20,scale_units='width',color='white',pivot='tail',\
                      width=3e-3,headwidth=5,ls='solid',linewidth=1.25,edgecolor='black',zorder=20)
        ax.set_title('Error')
        plt.tight_layout()
        plt.show()
        print(f'Mean x error: {np.round(np.mean(err_x[start[0]:stop[0]:sf,start[1]:stop[1]]),3)}')
        print(f'Mean y error: {np.round(np.mean(err_y[start[0]:stop[0]:sf,start[1]:stop[1]]),3)}')
        print(f'Mean z error: {np.round(np.mean(err_z[start[0]:stop[0],start[1]:stop[1]]),3)}')

    if plot_ana:
        limit = np.pi/2
        points = 21
        X,Y = np.meshgrid(np.linspace(-limit,limit,points), np.linspace(-limit,limit,points))
        QX,QY = np.meshgrid(np.arange(points),np.arange(points))
        #Ana_polx = np.cos(theta)*np.sin(2*X) + np.cos(2*X)*np.cos(phi)*np.sin(theta)  #wrong???
        #Ana_poly = -np.cos(2*X)*np.cos(theta)*np.sin(2*Y) + np.sin(theta)*(np.cos(phi)*np.sin(2*X)*np.sin(2*Y) + np.cos(2*Y)*np.sin(phi))
        #Ana_polz = np.cos(2*Y)*(np.cos(2*X)*np.cos(theta) - 2*np.cos(X)*np.cos(phi)*np.sin(X)*np.sin(theta) + np.sin(2*Y)*np.sin(theta)*np.sin(phi))
        Ana_polx = np.cos(theta)*np.sin(2*X) + np.cos(2*X)*np.cos(phi)*np.sin(theta)
        Ana_poly = -np.cos(2*X)*np.cos(theta)*np.sin(2*Y) + np.sin(theta)*(np.cos(phi)*np.sin(2*X)*np.sin(2*Y) + np.cos(2*Y)*np.sin(phi))
        Ana_polz = np.cos(2*Y)*np.cos(2*X)*np.cos(theta) - np.sin(theta)*(-np.cos(2*Y)*np.cos(phi)*np.sin(2*X) + np.sin(2*Y)*np.sin(phi))
        
        fig,ax = plt.subplots(1,1,figsize=(10,8))
        ax.set_title(f'Analytic: phi={round(phi,2)}, theta={round(theta,2)}')
        imz = ax.imshow(Ana_polz,cmap='GnBu',interpolation='none',origin='lower',zorder=0,vmin=-1,vmax=1)
        plt.colorbar(imz,ax=ax,shrink=.5)
        ax.quiver(QX,QY,Ana_polx,Ana_poly,scale=20,scale_units='width',color='white',pivot='tail',\
                  width=3e-3,headwidth=5,ls='solid',linewidth=1.25,edgecolor='black',zorder=20)
        plt.tight_layout()
        returned = returned + [fig,ax]
    return returned
    

### Square vs triangle

In [ ]:
polx.value = 1.0
poly.value = 0.0
polz.value = 0.0
square = do_run('testing','square',plot=True,ray_num=GOOD_RUN);
if True:
    polx.value = np.cos(np.pi/4)
    poly.value = np.sin(np.pi/4)
    polz.value = 0.0
    tri = do_run('testing','triangle',plot=True,ray_num=GOOD_RUN);

lims = square[0].metadata.limits  #detector extent

In [ ]:
sq_path = r"C://Users//samck//Documents//GitHub//OAM-HFIR-11-22//sim_data//testing//square_8"
polx,_,lims = read_run(sq_path,r"polx_det.dat")
poly,*_ = read_run(sq_path,r"poly_det.dat")
polz,*_ = read_run(sq_path,r"polz_det.dat")
square = np.array([polx,poly,polz])

tr_path = r"C://Users//samck//Documents//GitHub//OAM-HFIR-11-22//sim_data//testing//triangle_4"
polx,*_ = read_run(tr_path,r"polx_det.dat")
poly,*_ = read_run(tr_path,r"poly_det.dat")
polz,*_ = read_run(tr_path,r"polz_det.dat")
tri = np.array([polx,poly,polz])

In [ ]:
plot_mcstas(square,proj=True,plot_indx=0,detlims=lims)
plot_mcstas(square,texture=True,plot_indx=0,detlims=lims)
if True:
    plot_mcstas(tri,proj=True,plot_indx=0,detlims=lims)
    plot_mcstas(tri,texture=True,plot_indx=0,detlims=lims)

#### Fitting polarization components

In [ ]:
sq_pb,_ = get_beam_pol(square)
tr_pb,_ = get_beam_pol(tri)

try:
    points = square[0].metadata.dimension[0]  #pixel number
except:
    points = len(square[0])
x = np.linspace(lims[0],lims[1],points)
y = np.linspace(lims[2],lims[3],points)

X,Y = np.meshgrid(x,y)
xdata = np.vstack((X.ravel(),Y.ravel()))

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('|P(x,y)| Rect.')
plt.imshow(sq_pb,origin='lower',extent=np.array(lims)*1e3,vmin=.6,vmax=1)
plt.colorbar()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('|P(x,y)| Tri.')
plt.imshow(tr_pb,origin='lower',extent=np.array(lims)*1e3,vmin=.6,vmax=1)
plt.colorbar()
fig.tight_layout()
plt.show()

plt.plot(x,np.mean(sq_pb,axis=0))
plt.plot(x,np.mean(sq_pb,axis=1))
plt.grid()
plt.show()

plt.plot(x,np.mean(tr_pb,axis=0))
plt.plot(x,np.mean(tr_pb,axis=1))
plt.grid()
plt.show()

In [ ]:
theta,phi = np.pi/2,np.pi/4  #checking analytic beam polarization
limit = 7*np.pi/2*.90945  # 2 cm total
points = 256
X,Y = np.meshgrid(np.linspace(-limit,limit,points), np.linspace(-limit,limit,points))
QX,QY = np.meshgrid(np.arange(points),np.arange(points))
Ana_polx = np.cos(theta)*np.sin(2*X) + np.cos(2*X)*np.cos(phi)*np.sin(theta)
Ana_poly = -np.cos(2*X)*np.cos(theta)*np.sin(2*Y) + np.sin(theta)*(np.cos(phi)*np.sin(2*X)*np.sin(2*Y) + np.cos(2*Y)*np.sin(phi))
Ana_polz = np.cos(2*Y)*np.cos(2*X)*np.cos(theta) - np.sin(theta)*(-np.cos(2*Y)*np.cos(phi)*np.sin(2*X) + np.sin(2*Y)*np.sin(phi))

ana_pol,_ = get_beam_pol([Ana_polx,Ana_poly,Ana_polz])

plt.imshow(ana_pol,origin='lower')
plt.colorbar()
plt.show()

plot_mcstas([Ana_polx,Ana_poly,Ana_polz],texture=True,plot_indx=0,detlims=lims)

In [ ]:
def cos_2d(xy,ax,kx,ky,phx,phy):
    '''2D cosine fitting for use with curve_fit from scipy.'''
    x, y = xy
    return ax*np.cos(kx*x + phx)*np.cos(ky*y + phy)

try:
    ydata = square[0].Intensity.ravel()
except:
    ydata = square[0].ravel()
guess = (1,2*np.pi/.003,2*np.pi,0,0)

parms_x,_ = curve_fit(cos_2d,xdata,ydata,p0=guess)
print(f'Amp, kx, ky, phix, phiy: \n {np.round(parms_x,3)}')

plt.plot(x,cos_2d((X,Y),*guess)[134,:],color='black')
plt.plot(x,cos_2d((X,Y),*parms_x)[134,:],color='C0')
#plt.plot(x,square[0].Intensity[134,:],color='C0')
plt.grid()
plt.show()

plt.plot(x,cos_2d((X,Y),*guess)[:,134],color='black')
plt.plot(x,cos_2d((X,Y),*parms_x)[:,134],color='C1')
#plt.plot(x,square[0].Intensity[:,134],color='C1')
plt.grid()
plt.show()

try:
    plt.imshow(square[0].Intensity,origin='lower',extent=lims)
except:
    plt.imshow(square[0],origin='lower',extent=lims)
plt.colorbar()
plt.show()

plt.imshow(soln_x:=cos_2d((X,Y),*parms_x),origin='lower',extent=lims)
plt.colorbar()
plt.show()

try:
    plt.imshow(soln_x-square[0].Intensity,origin='lower',extent=lims,vmin=-.1,vmax=.1)
except:
    plt.imshow(soln_x-square[0],origin='lower',extent=lims,vmin=-.1,vmax=.1)
plt.colorbar()
plt.show()

In [ ]:
def cos_2d(xy,ax,kx,ky,phx,phy):
    '''2D cosine fitting for use with curve_fit from scipy.'''
    x, y = xy
    return ax*np.cos(kx*x + phx)*np.cos(ky*y + phy)

try:
    ydata = square[1].Intensity.ravel()
except:
    ydata = square[1].ravel()
guess = (1,2*np.pi/.003,2*np.pi/.0022,-np.pi/2,np.pi/2)

parms_y,_ = curve_fit(cos_2d,xdata,ydata,p0=guess)
print(f'Amp, kx, ky, phix, phiy: \n {np.round(parms_y,3)}')

plt.plot(x,cos_2d((X,Y),*guess)[134,:],color='black')
plt.plot(x,cos_2d((X,Y),*parms_y)[134,:],color='C0')
#plt.plot(x,square[1].Intensity[134,:],color='C0')
plt.grid()
plt.show()

plt.plot(x,cos_2d((X,Y),*guess)[:,134],color='black')
plt.plot(x,cos_2d((X,Y),*parms_y)[:,134],color='C1')
#plt.plot(x,square[1].Intensity[:,134],color='C1')
plt.grid()
plt.show()

try:
    plt.imshow(square[1].Intensity,origin='lower',extent=lims)
except:
    plt.imshow(square[1],origin='lower',extent=lims)
plt.colorbar()
plt.show()

plt.imshow(soln_y:=cos_2d((X,Y),*parms_y),origin='lower',extent=lims)
plt.colorbar()
plt.show()

try:
    plt.imshow(soln_y-square[1].Intensity,origin='lower',extent=lims,vmin=-.1,vmax=.1)
except:
    plt.imshow(soln_y-square[1],origin='lower',extent=lims,vmin=-.1,vmax=.1)
plt.colorbar()
plt.show()

In [ ]:
def cos_2d(xy,ax,kx,ky,phx,phy):
    '''2D cosine fitting for use with curve_fit from scipy.'''
    x, y = xy
    return ax*np.cos(kx*x + phx)*np.cos(ky*y + phy)

try:
    ydata = square[2].Intensity.ravel()
except:
    ydata = square[2].ravel()
guess = (1,2*np.pi/.003,2*np.pi/.0022,-3*np.pi/2,0)

parms_z,_ = curve_fit(cos_2d,xdata,ydata,p0=guess)
print(f'Amp, kx, ky, phix, phiy: \n {np.round(parms_z,3)}')

plt.plot(x,cos_2d((X,Y),*guess)[134,:],color='black')
plt.plot(x,cos_2d((X,Y),*parms_z)[134,:],color='C0')
#plt.plot(x,square[2].Intensity[134,:],color='C0')
plt.grid()
plt.show()

plt.plot(x,cos_2d((X,Y),*guess)[:,134],color='black')
plt.plot(x,cos_2d((X,Y),*parms_z)[:,134],color='C1')
#plt.plot(x,square[2].Intensity[:,134],color='C1')
plt.grid()
plt.show()

try:
    plt.imshow(square[2].Intensity,origin='lower',extent=lims)
except:
    plt.imshow(square[2],origin='lower',extent=lims)
plt.colorbar()
plt.show()

plt.imshow(soln_z:=cos_2d((X,Y),*parms_z),origin='lower',extent=lims)
plt.colorbar()
plt.show()

try:
    plt.imshow(soln_z-square[2].Intensity,origin='lower',extent=lims,vmin=-.1,vmax=.1)
except:
    plt.imshow(soln_z-square[2],origin='lower',extent=lims,vmin=-.1,vmax=.1)
plt.colorbar()
plt.show()

#### Fitting beam polarization

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Beam pol: |P(x,y)|')
plt.imshow(sq_pb,origin='lower',extent=lims,vmax=.9,vmin=.6)
plt.colorbar()
plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Beam pol using fits: |P(x,y)|')
plt.imshow(fits_pb:=np.sqrt(soln_x**2 + soln_y**2 + soln_z**2),origin='lower',extent=lims,vmax=.9,vmin=.6)
plt.colorbar()
plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Difference')
plt.imshow(sq_pb-fits_pb,origin='lower',extent=lims,vmax=.1,vmin=-.1)
plt.colorbar()
plt.show()

print(f'Amp, kx, ky, phix, phiy: ')
print(parms_x)
print(parms_y)
print(parms_z)

In [ ]:
def cos_2d_os(xy,ax,kx,ky,phx,phy,os):
    '''2D cosine with offset for fitting beam polarization.'''
    x, y = xy
    return ax*np.cos(kx*x + phx)*np.cos(ky*y + phy) + os

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Beam pol using single fits')
plt.imshow(fits_pb,origin='lower',extent=lims)
plt.colorbar()
plt.show()

ydata = fits_pb.ravel()
guess = (.05,4e3,1,0,0,.82)

parms_PB,_ = curve_fit(cos_2d_os,xdata,ydata,p0=guess)
print(f'Amp, kx, ky, phix, phiy: \n {np.round(parms_PB,3)}')

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Direct fitted Beam pol')
plt.imshow(pb_direct:=cos_2d_os((X,Y),*parms_PB),origin='lower',extent=lims)
plt.colorbar()
plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Difference')
plt.imshow(fits_pb-pb_direct,origin='lower',extent=lims,vmax=.1,vmin=-.1)
plt.colorbar()
plt.show()

In [ ]:
def cos_2d_os2(xy,A,B,p):
    '''2D cosine with offset for fitting beam polarization.'''
    x, y = xy
    return np.sqrt((A*np.cos(2*np.pi/p*x))**2 + (B*np.sin(2*np.pi/p*x))**2)

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Beam pol using single fits')
plt.imshow(fits_pb,origin='lower',extent=lims)
plt.colorbar()
plt.show()

ydata = fits_pb.ravel()
guess = (.9,.7,.003)

parms_PB,_ = curve_fit(cos_2d_os2,xdata,ydata,p0=guess)
print(f'A, B, p: \n {np.round(parms_PB,5)}')

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Direct fitted Beam pol')
plt.imshow(pb_direct:=cos_2d_os2((X,Y),*parms_PB),origin='lower',extent=lims)
plt.colorbar()
plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Difference')
plt.imshow(fits_pb-pb_direct,origin='lower',extent=lims,vmax=.1,vmin=-.1)
plt.colorbar()
plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Difference')
plt.imshow(fits_pb-sq_pb,origin='lower',extent=lims,vmax=.1,vmin=-.1)
plt.colorbar()
plt.show()

In [ ]:
A,B,kx,ky,x,y = sym.symbols(r'A,B,k_x,k_y,x,y',real=True)

px = A*sym.cos(kx*x)
py = B*sym.cos(kx*x + sym.pi/2)*sym.cos(ky*y + sym.Rational(3,2)*sym.pi)
pz = B*sym.cos(kx*x + sym.pi/2)*sym.cos(ky*y + sym.pi)

display(px,py,pz)

pb_ana = sym.sqrt(px**2 + py**2 + pz**2)
display(pb_ana)
display(pb_ana.expand().simplify())

pb_ana = pb_ana.subs({A:.878,B:.681,kx:2*np.pi/.003,ky:0.})

func = sym.lambdify([x,y],pb_ana,modules=['numpy'])

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Guess')
plt.imshow(func(X,Y),origin='lower',extent=lims)
plt.colorbar()
plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title(r'$|P_B(x,y)|$')
plt.imshow(sq_pb,origin='lower',extent=lims,vmax=.9,vmin=.6)
plt.colorbar()
ax.set_yticks(np.linspace(-.01,.01,7),np.round(np.linspace(-10.,10.,7),1))
ax.set_xticks(np.linspace(-.01,.01,7),np.round(np.linspace(-10.,10.,7),1))
ax.set_xlabel(r'$x$ [mm]')
ax.set_ylabel(r'$y$ [mm]')
plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Difference')
plt.imshow(sq_pb-func(X,Y),origin='lower',extent=lims)
plt.colorbar()
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title(r'$|P_B(x,y)|$')
plt.imshow(tr_pb,extent=lims*1e3,vmax=.9,vmin=.6)
plt.colorbar(shrink=.75)
ax.set_xlabel(r'$x$ [mm]')
ax.set_ylabel(r'$y$ [mm]')
plt.show()

plt.plot(first:=np.mean(tr_pb,axis=0))
plt.plot(second:=np.mean(tr_pb,axis=1))
plt.plot(first*second)
plt.grid()
plt.show()

In [ ]:
def cos_2d_os2(xy,A,B,p):
    '''2D cosine with offset for fitting beam polarization.'''
    x, y = xy
    return np.sqrt((A*np.cos(2*np.pi/p*x))**2 + (B*np.sin(2*np.pi/p*x))**2)

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('')
plt.imshow(tr_pb,origin='lower',extent=lims)
plt.colorbar()
plt.show()

ydata = tr_pb.ravel()
guess = (.9,.7,.003)

parms_PB,_ = curve_fit(cos_2d_os2,xdata,ydata,p0=guess)
print(f'A, B, p: \n {np.round(parms_PB,5)}')

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('')
plt.imshow(pb_direct:=cos_2d_os2((X,Y),*parms_PB),origin='lower',extent=lims)
plt.colorbar()
plt.show()

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title('Difference')
plt.imshow(pb_direct-tr_pb,origin='lower',extent=lims,vmin=-.1,vmax=.1)
plt.colorbar()
plt.show()

### Scanning the first nutator

In [ ]:
nu_scan = []
for phi in np.linspace(0.,np.pi/2,10):
    polx.value = np.cos(phi)
    poly.value = np.sin(phi)
    nu_scan.append(do_run('Triangles',f'phi_{round(phi,1)}'))
    plot_mcstas(nu_scan[-1],texture=True)

In [ ]:
for n in nu_scan:
    plot_mcstas(n,proj=True)

In [ ]:
for phi in [0]:
    for n in [nu_scan[len(nu_scan)//2]]:
        plot_mcstas(n,proj=True,nu_post_angle=1)

In [ ]:
plot_mcstas(nu_scan[len(nu_scan)//2],texture=True,nu_post_angle=np.pi/4)

***
## Experimental Data Analysis:
***

#### Analysis functions

Contains functions to load the Timepix Data into this notebook, calculate the polarization, and fit the experimental data scans.

In [ ]:
directories = {'work':'C://Users//xsm//Desktop//SEMSANS_OAM//',\
               'home':'G://SCGSR_data//All_timepix_data//'} #'laptop_1':'C://Users//samck//Documents//Data//'}

spice_directs = {'work':' ',\
                'home':r"G://SCGSR_data//HB2D_9_7//SpICE//User//",\
                'laptop':' '}

for _,s in directories.items():
    try:
        if isdir(s): DEFAULT_DIRECT = s
    except Exception: pass
assert isdir(DEFAULT_DIRECT), 'Default data path not found!'
print(f'Default directory: {DEFAULT_DIRECT}')

for _,s in spice_directs.items():
    try:
        if isdir(s): DEFAULT_SPICE_LOG_DIRECT = s
    except Exception: pass
assert isdir(DEFAULT_SPICE_LOG_DIRECT), 'Default SpICE log directory is not found!'
print(f'Default spice log directory: {DEFAULT_SPICE_LOG_DIRECT}')

def fumi_reader(run_num,direct=DEFAULT_DIRECT,plot=False,ROI_x=(100,400),ROI_y=(100,400),print_run=False,extent=None):
    """Reads and plots the csv datafile. Can specify a region of interest with ROI.
    Returns tuple of reduced data and run number."""
    filename = direct + f'Run_{run_num}_DSCALE_2_ROT_90.0.txt'
    if print_run: print(f'Run number used: {run_num}')
    assert isdir(direct), f'Data directory {direct} is not found!'
    assert isfile(filename), f'Datafile {filename} is not found!'
    
    with open(filename) as f:
        lines = csv.reader(f,delimiter=',')
        data = np.array([l for l in lines],dtype=float)[ROI_x[0]:ROI_x[1],ROI_y[0]:ROI_y[1]].T
    if plot: ShowImage(data,title=f'Run {run_num}',extent=extent)
    return data,run_num

def ShowImage(inImage,vmin=0,vmax=None,title='',extent=None):
    """Plots image in reduction function."""
    if vmax==None: vmax=np.max(inImage)
    fig, ax = plt.subplots(1,1,figsize=(8,6))
    ax.set_title(title)
    im = ax.imshow(inImage,origin="lower",aspect="auto",vmin=vmin,vmax=vmax,extent=extent)
    fig.colorbar(im)
    fig.tight_layout()
    plt.show()

def get_pol(up,down,zero=True,plot=False,extent=None):
    """Returns the polarization, run numbers used, and errors."""
    u,u_rn = up
    d,d_rn = down
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  #ignores nan mulitply and zero divide warnings
        pol = (u - d)/(u + d)
        errs = 2/(u + d)**2*np.sqrt(u**2*d + u*d**2)
    if zero:
        pol = np.nan_to_num(pol,0.0)
        errs = np.nan_to_num(errs,.5)      

    if plot:
        fig,ax = plt.subplots(nrows=1,ncols=2,figsize=(12,6))
        im1 = ax[0].imshow(pol,origin='lower',extent=extent)#,vmin=-1,vmax=1)
        ax[0].set_title('Pol')
        plt.colorbar(im1,ax=ax[0],shrink=.5)
        
        ax[1].set_title('Error')
        im2 = ax[1].imshow(errs,origin='lower',vmin=0,vmax=.5,extent=extent)
        plt.colorbar(im2,ax=ax[1],shrink=.5)
        plt.suptitle(f'Runs {u_rn} and {d_rn}')
        plt.tight_layout()
        plt.show()
    
    return pol,errs,u_rn,d_rn

def cross(run,errs,X,Y,plot=True,vmin=-1):
    """Analyzes selected cross for each scan. Need to check errors are
    being calculated correctly."""
    h = np.sum(run[Y,:],axis=0)/len(Y)
    v = np.sum(run[:,X],axis=1)/len(X)

    try:
        h_err = np.sum(errs[Y,:],axis=0)/len(Y)
        v_err = np.sum(errs[:,X],axis=1)/len(X)
    except:
        print('Error calculation failed!')
        h_err = np.zeros(len(h))
        v_err = np.zeros(len(v))
    
    if plot:
        fig,ax = plt.subplots(1,2,figsize=(19,6))
        im = ax[0].imshow(run,origin='lower',interpolation='None',vmin=vmin,vmax=1)
        plt.colorbar(im,ax=ax[0],shrink=.8)
        for y in (Y[0],Y[-1]):
            ax[0].axhline(y,linestyle='--',color='red')
        for x in (X[0],X[-1]):
            ax[0].axvline(x,linestyle='-.',color='black')
        ax[1].plot(h,label='Horz.',color='red',ms=7.5,zorder=10,ls='--',linewidth=2.5)
        ax[1].errorbar(range(len(h)),h,h_err,ls='none',zorder=5,color='red')
        
        ax[1].plot(v,label='Vert.',color='black',ms=7.5,zorder=10,ls='-.',linewidth=2.5)
        ax[1].errorbar(range(len(v)),v,v_err,ls='none',zorder=5,color='black')
        #ax[1].legend()
        ax[1].grid(True)

    return (h,v,h_err,v_err),fig,ax,im

def get_ps(data,plot=False,show=False,niels_cutoff=.1,lim=15):
    """Returns power spectrum of the data, plots it, and period from power spectrum."""
    N,M = np.shape(data)
    fh,fv = fftfreq(N),fftfreq(M)
    ft = fft2(data)
    ps = np.abs(ft**2)
    
    max = np.max(ps)
    fil = np.where(ps > max*niels_cutoff,ft,np.zeros(np.shape(ps)))
    points = np.array([(i,j) for (i,j) in np.argwhere(fil) if fh[i]>=0 and fv[j]>=0])  #only posi vibes
    
    freqs = np.array([[fh[i],fv[j]] for (i,j) in points])
    weights = np.array([ps[i,j] for (i,j) in points])/max    
    freq = np.array([np.sum(weights*freqs[:,i]) for i in range(np.shape(freqs)[1])])/np.sum(weights)
    fil = np.real(ifft2(fil))  #back to real space

    if show:
        print(f'Horizontal frequencies: {freqs[:,0]}')
        print(f'Vertical frequencies: {freqs[:,1]}')
        print(f'Weights: {weights}')
        print(f'Weighted frequencies: {freq}')
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  #ignores nan mulitply and zero divide warnings
            print(f'Dominant periods: {np.round(1/freq*pix_to_mm,2)} (mm)')
    if plot:
        fig, ax = plt.subplots(1,2,figsize=(12,6))
        ax[0].set_title(f'Log power spectrum')
        im = ax[0].imshow(np.log(ps)[:lim,:lim],origin='lower',extent=[fh[0],fh[lim],fv[0],fv[lim]])
        plt.colorbar(im,ax=ax[0],shrink=.75)
        ax[1].set_title(f'Niels filtered')
        im = ax[1].imshow(fil,origin='lower')
        plt.colorbar(im,ax=ax[1],shrink=.75)
        plt.tight_layout()
        plt.show()
        
    return ps,fil,freq

def reduce_scan(start,end,nu_scan=True,ROI_x=(100,400),ROI_y=(100,400),direct=DEFAULT_DIRECT):
    """Reduces scan to array of polarizations (or intensities) and errors.
    Doesn't include the end scan."""
    if nu_scan:
        up = np.sum([fumi_reader(i,direct=direct,ROI_x=ROI_x,ROI_y=ROI_y)[0] for i in range(start,end,2)],axis=0)
        down = np.sum([fumi_reader(i,direct=direct,ROI_x=ROI_x,ROI_y=ROI_y)[0] for i in range(start+1,end,2)],axis=0)
        pol,err,*_ = get_pol((up,start),(down,start))
    else:
        pol = np.sum([fumi_reader(i,direct=direct,ROI_x=ROI_x,ROI_y=ROI_y)[0] for i in range(start,end)],axis=0)   #not really a polarization
        err = np.sqrt(pol)  #counting error
    return pol,err

def reduce_ac_scan(start,end,nu_scan=True,ROI_x=(100,400),ROI_y=(100,400),direct=DEFAULT_DIRECT):
    """Reduces scan to array of polarizations (or intensities) and errors.
    Doesn't include the end scan."""
    pols,errs = [],[]
    if nu_scan:
        for i in range(start,end,2):
            up = fumi_reader(i,direct=direct,ROI_x=ROI_x,ROI_y=ROI_y)
            down = fumi_reader(i+1,direct=direct,ROI_x=ROI_x,ROI_y=ROI_y)
            pol,err,*_ = get_pol(up,down)
            pols.append(pol)
            errs.append(err)
    else:
        for i in range(start,end):
            ints,_ = fumi_reader(i,direct=direct,ROI_x=ROI_x,ROI_y=ROI_y)
            pols.append(ints)  #not really a polarization
            errs.append(np.sqrt(ints))  #counting error
    return np.array(pols),np.array(errs)

def spice_log_reader(scan,direct=DEFAULT_SPICE_LOG_DIRECT,show=False,scan_offset=-1000,only_useful=True):
    """Finds the scan parameters from the Spice logfile.txt file.
    Returns scan number, scan title, date/time, and SpICE motor settings.
    Only working for the second OAM experiment as of (6/11/24)."""
    assert isdir(direct), 'Spice logfile.txt directory was not found!'
    assert isfile(direct+"logfile.txt"), 'Spice logfile.txt was not found!'

    scan = int(scan + scan_offset)  #Timepix3 pc is 1000 scans ahead of the competition

    useful_settings = ('nu_pre','ac_1','up_1','down_1','up_2','down_2','ac_2','nu_post')  #depends on experiment
    
    with open(direct+"logfile.txt", "r") as log:
        lines = log.readlines()
        check_string = f'# scan = {scan}\n'
        found = False
        for indx,line in enumerate(lines):
            if line == check_string:
                found = True
                title = lines[indx+10][15:-1]  #line numbers determined by SpICE
                date = datetime.strptime(lines[indx+1][9:-1],'%m/%d/%Y').date()
                time = datetime.strptime(lines[indx+2][9:-1],'%I:%M:%S %p').time()
                full_time = datetime.combine(date,time)  #for matplotlib
                parameters = lines[indx+24].split()
                values = lines[indx+25].split()
                for indx,v in enumerate(values):
                    try: values[indx]=float(v)
                    except Exception: pass  #should fail if not a number
        if not found:
            print(f'Scan {scan} not found!')
            return None,None,None,None
        settings = dict(zip(parameters,values))
        if only_useful: 
            print()
            settings = {k:settings[k] for k in useful_settings if k in settings}
        if show:
            print(f'Scan: {scan}')
            print(f'Title: {title}')
            print(f'Date: {date}')
            print(f'Time: {time}')
            print(f'Settings: {settings} \n')
        return scan,title,full_time,settings

def data_sniffer(scan_start,direct=DEFAULT_DIRECT,show=True,limit=np.inf):
    """Checks if all data files have been uploaded to the SNS analysis cluster.
    Returns list of filenames and run numbers."""
    filenames = glob.glob(direct + '*.txt')  #finds all text files
    print(f'Files found: {len(filenames)}')
    filenames = [f.split('\\')[1] for f in filenames]
    scan_nums = np.array(sorted([int(f.split('_')[1]) for f in filenames]))
    scan_nums = scan_nums[scan_nums>=scan_start]  #ignoring first few scans
    
    differences = np.diff(scan_nums,append=scan_nums[-1]+1)
    skips = differences<=limit  #ignoring all the big skips
    differences = differences[skips]
    scan_nums = scan_nums[skips]

    missing = scan_nums[np.argwhere(differences>1)]  #doesn't include multiple skips yet
    print(f'Missing scans: {missing+1}')
    for m in missing:
        pass #_ = spice_log_reader(m,show=True)
    
    if show:
        plt.figure(figsize=(12,4))
        plt.plot(scan_nums,differences,'o')
        plt.xlabel('Scan')
        plt.ylabel('Difference')
        plt.grid()
        plt.show()
    
    return filenames,scan_nums,differences

Seems like all of the data was eventually uploaded to the analysis server? I though some was missing, but it looks like its all there. The Spice data reader also needs some work.

In [ ]:
_,_,d = data_sniffer(6000,limit=1,show=True)

In [ ]:
print(DEFAULT_SPICE_LOG_DIRECT)

start = 9000 #6065
rng = 1
for i in range(start,start+rng):
    _ = spice_log_reader(i,show=True,direct=DEFAULT_SPICE_LOG_DIRECT+r'//exp6//')

### Start of cycle 502 data
***

This first few days of this data analysis are being done retroactively as of 9/14. During the experiment, Fankang took over the role of data analysis for the 2D SEMSANS experiment while I did the data analysis for AER II at CG4B.

#### 8/4 Analyzing last night's ac_1 ac_2 scan at 10 amps

In [ ]:
_ = fumi_reader(6067,plot=True,ROI_x=(150,350),ROI_y=(100,400))

In [ ]:
up = fumi_reader(7044,plot=False,ROI_x=(100,350),ROI_y=(100,350))
down = fumi_reader(7045,plot=False,ROI_x=(100,350),ROI_y=(100,350))
pol,err,*_ = get_pol(up,down,plot=True,zero=False)
pol,err,*_ = get_pol(up,down,plot=False,zero=True)  #with nan_to_num
h,v,*_ = cross(pol,err,range(145,155),range(145,155),plot=True)
_ = get_ps(pol,plot=True,show=False,niels_cutoff=.01)

RX,RY = (100,350),(100,350)  #global region of interest

#### 8/5 Analyzing last night's transverse OAM scan after adjusting the analyzing v-cavity (to do)

#### 8/5 Analyzing last night's transverse OAM scan after adjusting the analyzing v-cavity (to do)

We decided to choose the position with the most counts thinking that it would probably be the position of highest contrast. However, we are probably going through the center of the vcavity, so there are 2 aberrations from the channels. We decided on the cav_trans position given by scans 6279-6280 from yesterday. (-9 mm on the old coordinates, NOT re-zeroed.)

#### 8/8 Accelerator coil scan fitting and processing (to do)

#### 8/9 Long longitudinal PNAS scan analysis

Decided that the ac_1 ac_2 scans don't have good enough statistics per pixel. Going to run some PNAS-quality scans at 15 amps: 3 hours for each spin component: 10 minute scans with 18 subscans. New runs begin at run number 6044 on SpICE (7044).

Although nu_post is in "longitudinal mode", we are measuring the horizontal transverse component of the polarization in this mode. Tonight after this scan is finished, we will rotate the nutator by 90 degrees and look at the two other components of the spin texture by choosing ac_2 0 and at the pi/2 mode.

Fankang fitted the period to 4.4788, so we want it to be 1/4 of a period.

In [ ]:
up = fumi_reader(7044,plot=False,ROI_x=(100,350),ROI_y=(100,351))
down = fumi_reader(7045,plot=False,ROI_x=(100,350),ROI_y=(100,351))
pol,err,*_ = get_pol(up,down,plot=True,zero=False)
pol,err,*_ = get_pol(up,down,plot=False,zero=True)  #with nan_to_num
h,v,*_ = cross(pol,err,range(152,157),range(122,127),plot=True)

RX,RY = (100,351),(100,351)  #global region of interest

In [ ]:
pol_x,err_x = reduce_scan(7044,7080,nu_scan=True,ROI_x=RX,ROI_y=RY)
_,fig,ax,im = cross(pol_x,err_x,range(152,153),range(152,153),plot=True)
ax[0].set_yticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[0].set_xticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[0].set_xlabel(r'$x$ [mm]')
ax[0].set_ylabel(r'$y$ [mm]')

ax[1].set_xticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[1].set_xlabel(r'$x$ or $y$ [mm]')
ax[1].set_ylabel(r'Polarization')
plt.show()

In [ ]:
print(DEFAULT_SPICE_LOG_DIRECT)
_ = spice_log_reader(7044,show=True,direct=DEFAULT_SPICE_LOG_DIRECT+r'//exp5//')

In [ ]:
print(L1,L2,L3,L4)
sem_cur = get_sem(15.0,L1,L2,show=True)
_ = get_ch(15.0,L1,L2,L3,L4,show=True)

print(sem_cur)

_ = get_period(15,sem_cur,show=True)
_ = get_period(15,16.15,show=True)

In [ ]:
np.pi/(4.62e-2*4.25*3.5*(16.15-15))

In [ ]:
16.15-15.0

In [ ]:
11.86-10.71

#### 8/9 Transverse PNAS scan analysis

In [ ]:
up = fumi_reader(7080,plot=False,ROI_x=(100,350),ROI_y=(100,350))
down = fumi_reader(7081,plot=False,ROI_x=(100,350),ROI_y=(100,350))
pol,err,*_ = get_pol(up,down,plot=True,zero=False)
pol,err,*_ = get_pol(up,down,plot=False,zero=True)  #with nan_to_num
h,v,*_ = cross(pol,err,range(145,155),range(145,155),plot=True)
#_ = get_ps(pol,plot=True,show=False,niels_cutoff=.5)

RX,RY = (100,351),(100,351)  #global region of interest

In [ ]:
pol_y,err_y = reduce_scan(7080,7116,nu_scan=True,ROI_x=RX,ROI_y=RY)
_,fig,ax,im = cross(pol_y,err_y,range(73,74),range(118,119),plot=True)
ax[0].set_yticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[0].set_xticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[0].set_xlabel(r'$x$ [mm]')
ax[0].set_ylabel(r'$y$ [mm]')

ax[1].set_xticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[1].set_xlabel(r'$x$ or $y$ [mm]')
ax[1].set_ylabel(r'Polarization')
plt.show()

In [ ]:
print(DEFAULT_SPICE_LOG_DIRECT)
_ = spice_log_reader(7080,show=True,direct=DEFAULT_SPICE_LOG_DIRECT+r'//exp5//')

In [ ]:
pol_z,err_z = reduce_scan(7116,7152,nu_scan=True,ROI_x=RX,ROI_y=RY)  #ac_2 in pi/2 mode
_,fig,ax,im = cross(pol_z,err_z,range(74,75),range(131,132),plot=True)
ax[0].set_yticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[0].set_xticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[0].set_xlabel(r'$x$ [mm]')
ax[0].set_ylabel(r'$y$ [mm]')

ax[1].set_xticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[1].set_xlabel(r'$x$ or $y$ [mm]')
ax[1].set_ylabel(r'Polarization')
plt.show()

In [ ]:
def temp_func(x,amp,phi,p):
    return amp*np.cos(x*2*np.pi/p + phi)

temp = pol_z[:,74]
tempx = np.linspace(-10.375,10.375,len(pol_z[:,74]))
plt.plot(tempx,temp)

guess = (.7,0,4)
plt.plot(tempx,temp_func(tempx,*guess))
parms,_ = curve_fit(temp_func,tempx,temp,p0=guess)
print(f'Amp, phix, px: \n {np.round(parms,3)}')

plt.plot(tempx,temp_func(tempx,*parms))
plt.grid()
plt.show()

temp = pol_z[131,:]
tempx = np.linspace(-10.375,10.375,len(pol_z[:,74]))
plt.plot(tempx,temp)

guess = (.7,-np.pi/2,3.6)
plt.plot(tempx,temp_func(tempx,*guess))
parms,_ = curve_fit(temp_func,tempx,temp,p0=guess)
print(f'Amp, phix, px: \n {np.round(parms,3)}')

plt.plot(tempx,temp_func(tempx,*parms))
plt.grid()
plt.show()

In [ ]:
print(DEFAULT_SPICE_LOG_DIRECT)
_ = spice_log_reader(7116,show=True,direct=DEFAULT_SPICE_LOG_DIRECT+r'//exp5//')

#### Assembling the PNAS pics

In [ ]:
_ = plot_st(-pol_x,-pol_y,pol_z,plot_z=True,plot_t=True,start=(0,0),stop=(250,250),sf=7,vals=(-.8,.8),plot_ana=False,\
                 errx=err_x,erry=err_y,errz=err_z,plot_err=False)

In [ ]:
fig,ax = plot_st(-pol_x,-pol_y,pol_z,plot_z=True,plot_t=True,start=(0,0),stop=(250,250),sf=7,vals=(-.8,.8),plot_ana=False,\
                 errx=err_x,erry=err_y,errz=err_z,plot_err=False)
ax.set_yticks(np.linspace(0,245,7),np.round(np.linspace(-10.375,10.375,7),1))
ax.set_xticks(np.linspace(0,245,7),np.round(np.linspace(-10.375,10.375,7),1))
ax.set_xlabel(r'$x$ [mm]')
ax.set_ylabel(r'$y$ [mm]')
plt.show()

In [ ]:
fig,ax = plot_st(-pol_x,-pol_y,pol_z,plot_z=True,plot_t=True,start=(10,20),stop=(90,100),sf=3,vals=(-.8,.8),plot_ana=False,\
                 errx=err_x,erry=err_y,errz=err_z,plot_err=False)
ax.set_xticks(np.linspace(10,88,7),np.round(np.linspace(-10.375+10*.083,-10.375+88*.083,7),1))
ax.set_yticks(np.linspace(20,98,7),np.round(np.linspace(-10.375+20*.083,-10.375+98*.083,7),1))
ax.set_xlabel(r'$x$ [mm]')
ax.set_ylabel(r'$y$ [mm]')
plt.show()

In [ ]:
_ = plot_st(pol_x,pol_y,pol_z,plot_z=True,plot_t=False,start=(100,100),stop=(145,145),sf=2,vals=(-.5,.5),plot_ana=True,\
       errx=err_x,erry=err_y,errz=err_z,plot_err=False)
plt.show()

There seems to be strange optical abberrations with central channel of the v-cavity. The intensity is higher but at the cost of really bad polarization...

In [ ]:
beam_pol = np.sqrt(pol_x**2+pol_y**2+pol_z**2)

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title(r'$|P_B(x,y)|$')
im = ax.imshow(beam_pol,origin="lower",aspect="auto",vmax=1,vmin=0)
fig.colorbar(im,shrink=.75)
ax.set_yticks(np.linspace(0,250,7),np.round(np.linspace(-10.375,10.375,7),1))
ax.set_xticks(np.linspace(0,250,7),np.round(np.linspace(-10.375,10.375,7),1))
ax.set_xlabel(r'$x$ [mm]')
ax.set_ylabel(r'$y$ [mm]')
fig.tight_layout()
plt.show()

_,fig,ax,im = cross(beam_pol,None,range(75,76),range(45,46),plot=True,vmin=0)
ax[0].set_yticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[0].set_xticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[0].set_xlabel(r'$x$ [mm]')
ax[0].set_ylabel(r'$y$ [mm]')

ax[1].set_xticks(np.linspace(0,251,7),np.round(np.linspace(-10.375,10.375,7),1))
ax[1].set_xlabel(r'$x$ or $y$ [mm]')
ax[1].set_ylabel(r'Polarization')
plt.show()

### Start of cycle 503 data
***

#### 8/29 Setup and tuning

In [ ]:
_ = fumi_reader(9015,plot=True,ROI_x=(150,350),ROI_y=(100,400))

In [ ]:
cav_trans = []
for i in range(9017,9026):
    v,_ = fumi_reader(i,plot=True,ROI_x=(150,350),ROI_y=(150,350))
    cav_trans.append(v)
cav_trans = np.array(cav_trans)
plt.plot(np.sum(cav_trans,axis=(1,2)))
plt.grid()
plt.show()

In [ ]:
plt.plot(np.sum(cav_trans,axis=(1,2)))

In [ ]:
cav_trans = []
for i in [9018,9021,9024]:
    v,_ = fumi_reader(i,plot=True,ROI_x=(150,350),ROI_y=(150,350))
    cav_trans.append(v)
cav_trans = np.array(cav_trans)
plt.plot(np.sum(cav_trans,axis=(1,2)))
plt.grid()
plt.show()

#### 8/29 v-cavity analyzer superscan

In [ ]:
temp_direct = "C://Users//samck//Desktop//rot_trans_superscan//"
cav_trans7p5 = []
for i in range(9036,9058,2):
    up = fumi_reader(i,plot=False,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
    down = fumi_reader(i+1,plot=False,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
    cav_trans7p5.append(get_pol(up,down,zero=True,plot=True)[0])
cav_trans7p5 = np.array(cav_trans7p5)

In [ ]:
for i in [9046,9048,9050]:
    _ = fumi_reader(i,plot=True,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)

In [ ]:
rot_range = np.round(np.linspace(-.5,.5,11),1)
for k in range(11):
    plt.plot(np.mean(cav_trans7p5[k],axis=0),label=f'{rot_range[k]}')
ax = plt.gca()
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid()
plt.show()
for k in range(11):
    plt.plot(np.mean(cav_trans7p5[k],axis=1),label=f'{rot_range[k]}')
ax = plt.gca()
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid()
plt.show()

In [ ]:
cav_trans15 = []
for i in range(9081,9098,2):
    up = fumi_reader(i,plot=False,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
    down = fumi_reader(i+1,plot=False,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
    cav_trans15.append(get_pol(up,down,zero=True,plot=True)[0])
cav_trans15 = np.array(cav_trans15)

In [ ]:
for i in [9089,9091]:
    _ = fumi_reader(i,plot=True,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)

In [ ]:
rot_range = np.round(np.linspace(-.4,.4,9),1)
for k in range(9):
    plt.plot(np.mean(cav_trans15[k],axis=0),label=f'{rot_range[k]}')
ax = plt.gca()
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid()
plt.show()
for k in range(9):
    plt.plot(np.mean(cav_trans15[k],axis=1),label=f'{rot_range[k]}')
ax = plt.gca()
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid()
plt.show()

In [ ]:
plt.plot(np.mean(cav_trans7p5[6],axis=0))
plt.plot(np.mean(cav_trans15[5],axis=0))
plt.grid()
plt.show()

In [ ]:
plt.plot(cav_trans7p5[6,:,100])
plt.plot(cav_trans15[5,:,100])
plt.grid()
plt.show()

In [ ]:
cav_trans22p5 = []
for i in range(9099,9117,2):
    up = fumi_reader(i,plot=False,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
    down = fumi_reader(i+1,plot=False,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
    cav_trans22p5.append(get_pol(up,down,zero=True,plot=True)[0])
cav_trans22p5 = np.array(cav_trans22p5)

In [ ]:
sev,_ = fumi_reader(9048,plot=True,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
sev2,_ = fumi_reader(9049,plot=True,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
fif,_ = fumi_reader(9091,plot=True,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
fif2,_ = fumi_reader(9092,plot=True,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
twen,_ = fumi_reader(9109,plot=True,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)
twen2,_ = fumi_reader(9110,plot=True,ROI_x=(150,350),ROI_y=(150,350),direct=temp_direct)

In [ ]:
def get_p(u,d):
    return (u-d)/(u+d)

plt.plot(get_p(np.sum(sev,axis=1),np.sum(sev2,axis=1)))
plt.plot(get_p(np.sum(fif,axis=1),np.sum(fif2,axis=1)))
plt.plot(get_p(np.sum(twen,axis=1),np.sum(twen2,axis=1)))
plt.grid()
plt.show()

#### 8/30 Overnight scan

In [ ]:
temp_direct = "C://Users//samck//Desktop//overnight 8_29//"

up = fumi_reader(9117,plot=False,ROI_x=(175,325),ROI_y=(175,325),direct=temp_direct)
down = fumi_reader(9118,plot=False,ROI_x=(175,325),ROI_y=(175,325),direct=temp_direct)
pol,err,*_ = get_pol(up,down,plot=True,zero=False)
pol,err,*_ = get_pol(up,down,plot=False,zero=True)  #with nan_to_num
h,v,*_ = cross(pol,err,range(100,105),range(100,105),plot=True)
_ = get_ps(pol,plot=True,show=False,niels_cutoff=.5)

RX,RY = (175,325),(175,325)  #global region of interest

In [ ]:
temp_direct = "C://Users//samck//Desktop//overnight 8_29//"
pol_y,err_y = reduce_scan(9117,9117+30,nu_scan=True,ROI_x=RX,ROI_y=RY,direct=temp_direct)
_ = cross(pol_y,err_y,range(100,105),range(100,105),plot=True)

In [ ]:
_ = get_ps(pol_y,plot=True,show=False,niels_cutoff=.01)

In [ ]:
_ = cross(pol_y,err_y,range(90,95),range(105,110),plot=True)

In [ ]:
temp_direct = "C://Users//samck//Desktop//overnight 8_29//"
pol_y,err_y = reduce_scan(9117+30,9117+60,nu_scan=True,ROI_x=RX,ROI_y=RY,direct=temp_direct)
_ = cross(pol_y,err_y,range(100,105),range(100,105),plot=True)

#### 8/30 Day scan of uber triangle

In [ ]:
temp_direct = "C://Users//samck//Desktop//8_30 uber triangle//"

pol_y,err_y = reduce_scan(9181,9187,nu_scan=True,ROI_x=RX,ROI_y=RY,direct=temp_direct)
_ = cross(pol_y,err_y,range(50,52),range(50,52),plot=True)

#### 8/31-9/1 Day scan of uber triangle (redo)

For some reason, the Timepix was only collecting 10% of the counts, so all of the old data from this cycle will be poor. Also, the local storage on the timepix pc ran out space, causing the scan to stall yesterday at 5 pm. We still have two projections of the uber triangle, which may be enough. The scans seem to be going well, I'll finish the tranvsere PNAS run and throw in a cheeky ac_1 ac_2 scan in this same geometry, finishing with another longitudinal ac_1 ac_2 scan tomorrow morning.

After that, we'll try a different shape, maybe rhomohedral or rectangular? Probably will have until next Wednesday, so that's 5 days after this planned scans. May also want to try the different v-cavity channels.

In [ ]:
temp_direct = "C://Users//samck//Desktop//Long_PNAS//"

up = fumi_reader(9259,plot=False,ROI_x=(160,340),ROI_y=(100,350),direct=temp_direct)
down = fumi_reader(9260,plot=False,ROI_x=(160,340),ROI_y=(100,350),direct=temp_direct)
pol,err,*_ = get_pol(up,down,plot=True,zero=False)
pol,err,*_ = get_pol(up,down,plot=False,zero=True)  #with nan_to_num
h,v,*_ = cross(pol,err,range(145,155),range(145,155),plot=True)
_ = get_ps(pol,plot=True,show=False,niels_cutoff=.1)

RX,RY = (160,340),(100,350)  #global region of interest

In [ ]:
_,title,_,settings = spice_log_reader(9259,show=True)
_,title,_,settings = spice_log_reader(9288,show=True)

In [ ]:
pol_x,err_x = reduce_scan(9259,9289,nu_scan=True,ROI_x=RX,ROI_y=RY,direct=temp_direct)
_ = cross(pol_x,err_x,range(145,146),range(145,146),plot=True)

In [ ]:
temp_direct = "C://Users//samck//Desktop//Trans_PNAS//"

up = fumi_reader(9289,plot=False,ROI_x=(160,340),ROI_y=(100,350),direct=temp_direct)
down = fumi_reader(9290,plot=False,ROI_x=(160,340),ROI_y=(100,350),direct=temp_direct)
pol,err,*_ = get_pol(up,down,plot=True,zero=False)
pol,err,*_ = get_pol(up,down,plot=False,zero=True)  #with nan_to_num
h,v,*_ = cross(pol,err,range(145,155),range(145,155),plot=True)
_ = get_ps(pol,plot=True,show=False,niels_cutoff=.1)

RX,RY = (160,340),(100,350)  #global region of interest

In [ ]:
_,title,_,settings = spice_log_reader(9289,show=True)
_,title,_,settings = spice_log_reader(9318,show=True)

In [ ]:
pol_y,err_y = reduce_scan(9289,9319,nu_scan=True,ROI_x=RX,ROI_y=RY,direct=temp_direct)
_ = cross(pol_y,err_y,range(145,146),range(145,146),plot=True)

In [ ]:
_,title,_,settings = spice_log_reader(9319,show=True)
_,title,_,settings = spice_log_reader(9319+29,show=True)

In [ ]:
pol_z,err_z = reduce_scan(9319,9349,nu_scan=True,ROI_x=RX,ROI_y=RY,direct=temp_direct)
_ = cross(pol_z,err_z,range(145,146),range(145,146),plot=True)

In [ ]:
plot_st(pol_x,pol_y,pol_z.T,plot_z=True,plot_t=True,start=(0,0),stop=(150,150),sf=5,vals=(-.7,.7),plot_ana=False,\
       errx=err_x,erry=err_y,errz=err_z)

In [ ]:
plot_st(pol_x,pol_y,pol_z.T,plot_z=True,plot_t=True,start=(40,40),stop=(100,100),sf=2,vals=(-.7,.7),plot_ana=False,\
       errx=err_x,erry=err_y,errz=err_z)

In [ ]:
beam_pol = np.sqrt(pol_x**2+pol_y**2+pol_z**2)
ShowImage(beam_pol,vmin=None,vmax=None,title='Beam pol: |P(x,y)|')
_ = cross(beam_pol,None,range(150,155),range(195,200),plot=True,vmin=0)

#### 9/2 Checking the overnight scans

Seem to have a zero field crossing bececause I scanned nu_post instead of nu_pre. Also, there was a typo in the ac_1 ac_2 scan casuing the nu_post current to not change. Will redo this today and do the longitudinal mode tonight.

In [ ]:
_ = spice_log_reader(9475,show=True)

In [ ]:
for i in range(9475,9475+5):
    _ = spice_log_reader(i,show=True)
    up = fumi_reader(i,plot=True,ROI_x=(160,340),ROI_y=(100,350),direct='C:/Users/samck/Desktop/9_2 Check/')

In [ ]:
_ = spice_log_reader(9476,show=True)
_ = spice_log_reader(9559,show=True)

temp_direct = "C://Users//samck//Desktop//9_2 Check//"

up = fumi_reader(9476,plot=True,ROI_x=(160,340),ROI_y=(100,350),direct=temp_direct)
down = fumi_reader(9559,plot=True,ROI_x=(160,340),ROI_y=(100,350),direct=temp_direct)
pol,err,*_ = get_pol(up,down,plot=True,zero=False)
pol,err,*_ = get_pol(up,down,plot=False,zero=True)  #with nan_to_num
h,v,*_ = cross(pol,err,range(145,150),range(145,150),plot=True)
_ = get_ps(pol,plot=True,show=False,niels_cutoff=.5)

RX,RY = (160,340),(100,350)  #global region of interest

In [ ]:
_,title,_,settings = spice_log_reader(9468,show=True)

#### 9/2 Longitudinal mode check

In [ ]:
temp_direct = "C://Users//samck//Desktop//"

up = fumi_reader(9613,plot=True,ROI_x=(160,340),ROI_y=(100,350),direct=temp_direct)
down = fumi_reader(9614,plot=True,ROI_x=(160,340),ROI_y=(100,350),direct=temp_direct)
pol,err,*_ = get_pol(up,down,plot=True,zero=False)
pol,err,*_ = get_pol(up,down,plot=False,zero=True)  #with nan_to_num
h,v,*_ = cross(pol,err,range(145,150),range(145,150),plot=True)
_ = get_ps(pol,plot=True,show=False,niels_cutoff=.5)

RX,RY = (160,340),(100,350)  #global region of interest

#### 9/3 Rectangular spin texture (to do)

#### 9/5 Checking the triangle

It turns out that the adiabiticy from scanning the nutators in the triangular modes was okay. The issue was that ac_2 was not orthogonal to the nutator anymore, so there was some out of plane component of the polarization. Thankfully, the polarization data should be okay, but I need to think about how to extract the components. I think the only issue will be with the z component.

In [ ]:
up = fumi_reader(9735,plot=True,ROI_x=(160,340),ROI_y=(100,350))
down = fumi_reader(9736,plot=True,ROI_x=(160,340),ROI_y=(100,350))
pol,err,*_ = get_pol(up,down,plot=False,zero=True)  #with nan_to_num
h,v,*_ = cross(pol,err,range(145,150),range(145,150),plot=True)
_ = get_ps(pol,plot=True,show=False,niels_cutoff=.5)

RX,RY = (160,340),(100,350)  #global region of interest

In [ ]:
print(DEFAULT_SPICE_LOG_DIRECT)

start = 9735  #did pi on 2 first
for i in range(42):
    _ = spice_log_reader(start+i,show=True,direct=DEFAULT_SPICE_LOG_DIRECT+r'//exp6//')

In [ ]:
RX,RY = (150,300),(100,250)  #global region of interest

pol_x,err_x = reduce_scan(9755,9775,nu_scan=True,ROI_x=RX,ROI_y=RY)
_,fig,ax,im = cross(pol_x,err_x,range(79,80),range(80,81),plot=True)
plt.show()

In [ ]:
RX,RY = (160,321),(100,261)  #global region of interest

pol_x,err_x = reduce_scan(9755,9775,nu_scan=True,ROI_x=RX,ROI_y=RY)
_,fig,ax,im = cross(pol_x,err_x,range(79,80),range(80,81),plot=True)
ax[0].set_yticks(np.linspace(0,161,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[0].set_xticks(np.linspace(0,161,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[0].set_xlabel(r'$x$ [mm]')
ax[0].set_ylabel(r'$y$ [mm]')

ax[1].set_xticks(np.linspace(0,161,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[1].set_xlabel(r'$x$ or $y$ [mm]')
ax[1].set_ylabel(r'Polarization')
plt.show()

In [ ]:
#plt.plot(np.linspace(-6.64,6.64,len(pol_x)),pol_x[100,:])
plt.plot(np.linspace(-6.64,6.64,len(beam_pol)),beam_pol[100,:])
plt.grid()
#plt.plot(np.mean(beam_pol,axis=0))
#plt.plot(np.mean(beam_pol,axis=1))
plt.show()

In [ ]:
start = 9822  #did pi on 2 first
for i in range(30):
    print(spice_log_reader(start+i,show=False,direct=DEFAULT_SPICE_LOG_DIRECT+r'//exp6//')[:-2])

In [ ]:
pol_y,err_y = reduce_scan(9822,9842,nu_scan=True,ROI_x=RX,ROI_y=RY)
_,fig,ax,im = cross(pol_y,err_y,range(79,80),range(80,81),plot=True)
ax[0].set_yticks(np.linspace(0,161,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[0].set_xticks(np.linspace(0,161,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[0].set_xlabel(r'$x$ [mm]')
ax[0].set_ylabel(r'$y$ [mm]')

ax[1].set_xticks(np.linspace(0,161,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[1].set_xlabel(r'$x$ or $y$ [mm]')
ax[1].set_ylabel(r'Polarization')
plt.show()

In [ ]:
start = 9842  #did pi on 2 first
for i in range(30):
    print(spice_log_reader(start+i,show=False,direct=DEFAULT_SPICE_LOG_DIRECT+r'//exp6//')[:-2])

In [ ]:
pol_z,err_z = reduce_scan(9842,9862,nu_scan=True,ROI_x=RX,ROI_y=RY)
_,fig,ax,im = cross(pol_z,err_z,range(79,80),range(80,81),plot=True)
ax[0].set_yticks(np.linspace(0,161,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[0].set_xticks(np.linspace(0,161,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[0].set_xlabel(r'$x$ [mm]')
ax[0].set_ylabel(r'$y$ [mm]')

ax[1].set_xticks(np.linspace(0,161,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[1].set_xlabel(r'$x$ or $y$ [mm]')
ax[1].set_ylabel(r'Polarization')
plt.show()

In [ ]:
_ = spice_log_reader(9842,show=True,direct=DEFAULT_SPICE_LOG_DIRECT+r'//exp6//')

In [ ]:
np.pi/(4.62e-2*4.25*3.5*(21.54-20))

In [ ]:
np.pi/(4.62e-2*4.25*3.5*(22.14-20))

In [ ]:
_ = plot_st(pol_x,pol_y,pol_z,plot_z=True,plot_t=True,start=(0,0),stop=(160,160),sf=4,vals=(-.6,.6),plot_ana=False,\
            errx=err_x,erry=err_y,errz=err_z)
plt.show()

In [ ]:
fig,ax = plot_st(-pol_x,-pol_y,pol_z,plot_z=True,plot_t=True,start=(75,75),stop=(141,141),sf=2,vals=(-.6,.6),plot_ana=False,\
            errx=err_x,erry=err_y,errz=err_z,plot_err=False)
ax.set_xticks(np.linspace(75,139,7),np.round(np.linspace(-6.64+75*.083,-6.64+139*.083,7),1))
ax.set_yticks(np.linspace(75,139,7),np.round(np.linspace(-6.64+75*.083,-6.64+139*.083,7),1))
ax.set_xlabel(r'$x$ [mm]')
ax.set_ylabel(r'$y$ [mm]')
plt.show()

In [ ]:
beam_pol = np.sqrt(pol_x**2+pol_y**2+pol_z**2)
ShowImage(beam_pol,vmin=.3,vmax=.8,title='Beam pol: |P(x,y)|')

In [ ]:
beam_pol = np.sqrt(pol_x**2+pol_y**2+pol_z**2)

fig, ax = plt.subplots(1,1,figsize=(8,6))
ax.set_title(r'$|P_B(x,y)|$')
im = ax.imshow(beam_pol,origin="lower",aspect="auto",vmax=.8,vmin=.2)
fig.colorbar(im,shrink=.75)
ax.set_yticks(np.linspace(0,160,7),np.round(np.linspace(-6.64,6.64,7),1))
ax.set_xticks(np.linspace(0,160,7),np.round(np.linspace(-6.64,6.64,7),1))
ax.set_xlabel(r'$x$ [mm]')
ax.set_ylabel(r'$y$ [mm]')
fig.tight_layout()
plt.show()

_,fig,ax,im = cross(beam_pol,None,range(80,81),range(80,81),plot=True,vmin=None)
ax[0].set_yticks(np.linspace(0,160,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[0].set_xticks(np.linspace(0,160,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[0].set_xlabel(r'$x$ [mm]')
ax[0].set_ylabel(r'$y$ [mm]')

ax[1].set_xticks(np.linspace(0,160,7),np.round(np.linspace(-6.64,6.64,7),1))
ax[1].set_xlabel(r'$x$ or $y$ [mm]')
ax[1].set_ylabel(r'Polarization')
plt.show()

In [ ]:
plt.plot(np.linspace(-6.64,6.64,len(beam_pol)),beam_pol[100,:])
#plt.plot(np.mean(beam_pol,axis=0))
#plt.plot(np.mean(beam_pol,axis=1))
plt.show()

#### 9/5 Finding the period of ac_1

In [ ]:
start = 9775  #did pi on 2 first
for i in range(30):
    print(spice_log_reader(start+i,show=False)[:-2])

In [ ]:
temp_direct = 'C://Users//samck//Desktop//ac_1_scan//'
RX,RY = (180,320),(140,340)
ac1_scan = reduce_ac_scan(9972,9994,ROI_x=RX,ROI_y=RY,direct=temp_direct)
print(np.shape(ac1_scan))

In [ ]:
ac_currents = np.linspace(-2.5,2.5,11)
H,V = [],[]
for indx,a in enumerate(ac1_scan):
    #h,v,*_ = cross(a,None,range(52,57),range(92,97),plot=True);
    h,v,*_ = cross(a,None,range(52,57),range(100,105),plot=True);
    H.append(h)
    V.append(v)

In [ ]:
x = np.linspace(-2.5,2.5,11)

plt.imshow(np.sum(ac1_scan,axis=1),origin='lower',aspect='auto')
plt.show()
plt.plot(x,np.sum(ac1_scan,axis=1).mean(axis=1))
#plt.show()

f = lambda x,amp,freq,phase: cos(x,amp,freq,phase,0,0)
y = np.sum(ac1_scan,axis=1).mean(axis=1)

parms,_ = curve_fit(f,x,y)
plt.plot(x,f(x,*parms))
plt.show()
print(parms)